# SAC arrival_v2 — history k=12 cross-seed 3rd anchor (seed=7) on single_cross_s0 (1M, vanilla)

**Pre-context（基于 §7.9.2 k=12 seed=42 PASS-PLATEAU + §7.9.2' k=12 seed=0 CROSS-SEED-RESCUE，闭合 §7.9.6 唯一 open disclaimer）**：[`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) §7.9.2 当前 single_cross_s0 vanilla SAC 完整画像（8 runs）：

| run                                | seed | k    | final  | peak@step    | mean39 | OOB    | n_succ | Gate         |
|---                                 |---:  |---:  |---:    |---           |---:    |---:    |---:    |---           |
| §7.6.4 s0_k4                       | 42   | 4    | 0.100  | 0.367 @ 975k | 0.221  | 0.667  | 35/39  | FAIL floor   |
| §7.7.1 s0_k4 sister                | 0    | 4    | 0.400  | 0.533 @ 625k | 0.218  | 0.200  | 34/39  | FAIL         |
| **§7.8 s0_k8 anchor**              | 42   | 8    | **0.900** | 0.900 @ 475k | **0.636** | **0.100** | 37/39 | **PASS 5/5** |
| §7.9.1' s0_k8                      | 0    | 8    | 0.500  | 0.500 @ 925k | 0.260  | 0.133  | 32/39  | PARTIAL 2/5  |
| §7.9.1'' s0_k8                     | 7    | 8    | 0.867  | 0.900 @ 550k | 0.518  | 0.133  | 31/39  | BORDER 4/5   |
| **§7.9.2 s0_k12 anchor**           | 42   | 12   | **0.900** | 0.900 @ **375k** | 0.652 | **0.100** | 35/39 | **PASS-PLATEAU 5/5** |
| **§7.9.2' s0_k12 RESCUE**          | 0    | 12   | **0.900** | 0.900 @ 525k | 0.525 | **0.100** | 32/39 | **CROSS-SEED-RESCUE 5/5 ⭐** |
| **本 notebook s0_k12 3rd anchor**  | **7**| 12   | ?      | ?            | ?      | ?      | ?      | ?            |

**为什么补 seed=7（与原 2026-05-19 不补决定的对照）**：
- 原决定（[`docs/online_rl_line_summary.md`](../docs/online_rl_line_summary.md) §4.4 / arrival_v2 report §7.9.6）：现有 2/2 strict PASS + σ_final=0.064 已 thesis-acceptable，第 3 seed 边际信息量 < 2.5h L4 成本。
- 重审决定（2026-05-23+，paper revision / rebuttal 风险对冲）：2 seed σ_final estimate 在审稿人面前无自由度做 CI；3 seed 提供 sample σ + 真正闭合 §7.9.6 唯一 open disclaimer。失败风险接近零（seed=7 在 k=8 上已 BORDERLINE-PASS 4/5，OOB 卡线 1 ep；按 §7.9.3 单调相位跃迁逻辑，k=12 几乎 deterministic PASS）。

**关键 open questions（本 notebook 必须区分）**：

| Hypothesis | 含义 | 本 cell 期望 |
|---|---|---|
| **H_thesis-closure: k=12 三 seeds 全 strict PASS** | k=12 是 cross-seed sweet spot；3-seed σ_final ≤ 0.10 thesis-grade | final ≥ 0.85 AND OOB ≤ 0.10 |
| **H_partial-closure: final 闭合但 OOB residual** | k=8→k=12 解 final 不解 OOB 末段 noise；OOB residual 是 SAC variance 不是 history capacity | final ≥ 0.85 AND 0.10 < OOB ≤ 0.135 |
| **H_overstale: k=12 在 seed=7 上 over-stale 反害** | over-stale history 把 k=8 BORDER 推向 PARTIAL | final < k=8 s7 − 0.05 (= 0.817) |
| **H_collapse: severe regression** | k=12 在 seed=7 上 collapse；需 audit | final < 0.40 |

**本 notebook 任务（pure vanilla + history k=12，单变量 seed swap from k=12 seed=42 anchor 或 seed=0 sister）**：

| 维度                  | §7.9.2 k=12 anchor (seed=42) / §7.9.2' (seed=0) | 本 notebook (k=12, seed=7) |
|---                    |---                                              |---                          |
| `--seed`              | 42 / 0                                          | **7** ← 唯一变量            |
| `--history-length`    | 12                                              | 12                          |
| algorithm             | vanilla SAC                                     | vanilla SAC                 |
| sensor layout         | s0 (DVL-only)                                   | s0 (DVL-only)               |
| reward                | arrival_v2                                      | arrival_v2                  |
| flow U / target       | 1.5 / 1.5                                       | 1.5 / 1.5                   |
| total_steps           | 1M                                              | 1M                          |
| num_envs              | 6                                               | 6                           |
| benchmark             | `single_u15_cross_tgt15`                        | 同                          |
| obs_dim               | 144 (12×12)                                     | 144                         |

**Seed 选择 = 7 的理由**：与 §7.9.1'' k=8 seed=7 sister 配对，构成 same-seed 跨 history `k ∈ {8, 12} × seed=7` 二元 paired contrast；同时是 §7.9.2 k=12 矩阵的第 3 anchor（继 seed=42 / seed=0），用于计算 thesis-grade 3-seed σ_final。注意 k=4 seed=7 未测（per §7.9.6 pointer 不补），本 cell seed=7 跨 history 轨迹仅含 k=8 / k=12 两点；seed=0 跨 k=4/8/12 三点轨迹作为参考保留在 verdict 输出里。

**Gate**（与 §7 / §7.6 / §7.7 / §7.8 / §7.9 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Cross-seed closure verdict 规则（5-tier，沿用 §7.9.2 build_k12_seed0.py 5-tier schema，仅 main contrast 改为 same-seed k=8 s7）**：

| Verdict | 触发条件 | thesis 主张含义 |
|---|---|---|
| **CROSS-SEED-CLOSURE-3ANCHOR** | k=12 s7 strict 5/5 PASS（final≥0.85 AND OOB≤0.10）| 强证据 H_thesis-closure。§7.9.2 主张升格「k=12 三 seeds 全 strict PASS, 3-seed σ_final ≤ 0.10 thesis-grade」；§7.9.6 唯一 open disclaimer 关闭；paper rebuttal 可直接 cite |
| **CROSS-SEED-BORDERLINE-CLOSURE** | final ≥ 0.85 AND 0.10 < OOB ≤ 0.135 | H_partial-closure。final 维度完全闭合 (vs k=8 s7 Δfinal ≥ 0)；但 OOB ≈ 0.133 与 §7.9.1'' k=8 s7 BORDER 一致，跨 history 不变 → **OOB residual variance 与 actor information capacity 解耦**；motivates §8 P0 variance reduction 但 history 路径仍 thesis-defensible |
| **UNEXPECTED-DECAY** | k=12 s7 final < k=8 s7 final − 0.05 (=0.817) OR 0.40 ≤ final < 0.50 | H_overstale 部分支持。§7.9.2 主张回退「k=12 cross-seed 不普适；seed=7 上 over-stale 反害」；与 §7.9.2 (s42) / §7.9.2' (s0) PASS 直接冲突，需 audit |
| **PERSISTENT-STALL** | 0.50 ≤ final < 0.85 AND \|final − k=8 s7 final\| ≤ 0.20 | seed=7 stall 跨 history 持续。与 §7.9.2/2' k=12 PASS cross-seed 冲突；§8 P0 variance reduction 必要 |
| **COLLAPSE** | final < 0.40 | H_collapse。audit trainer_state grad-norm / loss curve / obs_dim 排除 training bug |

**输出根（与 §7.9.2 k=12 seed=42 / §7.9.2' k=12 seed=0 互不覆盖）**：
- `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_7/`

**总预算**：~2.5h L4（1 Colab Pro+ session；obs_dim 144 与 §7.9.2 一致；与 seed=42 / seed=0 同口径）。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑，与 §7.8 / §7.9.2 一致。


## 0. GPU sanity


In [1]:
!nvidia-smi | head -10


Fri May 22 17:24:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |


## 1. Mount Drive + cwd


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. Config — single phase（vanilla SAC + history k=12 + seed=7，与 §7.9.2 k=12 anchor 仅差一个 flag value）


In [3]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== SAC / env config (与 §7.9.2 k=12 anchor / §7.9.2' k=12 sister 严格一致，除 seed 外) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 12
TARGET_SPEED = 1.5
SEED = 7                                # ← 唯一与 §7.9.2 (seed=42) / §7.9.2' (seed=0) 不同；3rd anchor

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

# 显式拒绝所有 SAC 改进项 — 与 §7.8 / §7.9.2 一致
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10
BORDERLINE_OOB_RATE = 0.135  # BORDERLINE tier 上限

# Flow file（与 §7 / §7.9.2 全套严格一致）
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Single phase — single_cross s0 k=12 seed=7 (X_*) ====
X_BENCHMARK_KEY = 'single_u15_cross_tgt15'
X_TASK_GEOMETRY = 'cross_stream'
X_FLOW_PATH = SINGLE_FLOW
X_TOTAL_STEPS = 1_000_000
X_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_7')
X_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_7')
X_MANIFEST_PATH = Path(f'benchmarks/{X_BENCHMARK_KEY}.json')

# Baselines（cross-seed closure 需要全 8-run 对照）
X_K12_S42_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_42')  # §7.9.2 anchor PASS-PLATEAU
X_K12_S0_ROOT  = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_0')   # §7.9.2' CROSS-SEED-RESCUE
X_K8_S42_ROOT  = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42')   # §7.8 anchor PASS
X_K8_S0_ROOT   = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_0')    # §7.9.1' PARTIAL
X_K8_S7_ROOT   = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_7')    # §7.9.1'' BORDER (主对照 — same seed)
X_K4_S42_ROOT  = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')   # §7.6.4 floor
X_K4_S0_ROOT   = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_0')    # §7.7.1 sister
X_S1_UPPER_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')  # §7.1 upper

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT          = {PROBE_LAYOUT}')
print(f'OBJECTIVE             = {OBJECTIVE}')
print(f'HISTORY_LENGTH        = {HISTORY_LENGTH}')
print(f"SEED                  = {SEED}        ← 3rd anchor; 与 §7.9.2 (seed=42) / §7.9.2' (seed=0) 不同")
print(f'NUM_ENVS              = {NUM_ENVS}')
print(f'USE_ASYMMETRIC_CRITIC = {USE_ASYMMETRIC_CRITIC}')
print(f'USE_LAYERNORM         = {USE_LAYERNORM}')
print(f'UPDATES_PER_STEP      = {UPDATES_PER_STEP}')
print(f'DROPOUT_RATE          = {DROPOUT_RATE}')
print()
print(f'benchmark             : {X_BENCHMARK_KEY}')
print(f'geometry              : {X_TASK_GEOMETRY}')
print(f'total_steps           : {X_TOTAL_STEPS:,}')
print(f'expected obs_dim      : 12 * {HISTORY_LENGTH} = {12 * HISTORY_LENGTH} (per train_config.txt across all baselines)')
print(f'run_root              : {X_RUN_ROOT}')
print(f'§7.9.2  k=12 s42 PASS : {X_K12_S42_ROOT}')
print(f"§7.9.2' k=12 s0  RES  : {X_K12_S0_ROOT}")
print(f'§7.8    k=8  s42 PASS : {X_K8_S42_ROOT}')
print(f"§7.9.1' k=8  s0  PART : {X_K8_S0_ROOT}")
print(f"§7.9.1'' k=8 s7  BORD : {X_K8_S7_ROOT}    ← same-seed main contrast")
print(f'§7.6.4  k=4  s42 floor: {X_K4_S42_ROOT}')
print(f'§7.7.1  k=4  s0  sister: {X_K4_S0_ROOT}')
print(f'§7.1    s1 k4 s42 upper: {X_S1_UPPER_ROOT}')


PROBE_LAYOUT          = s0
OBJECTIVE             = arrival_v2
HISTORY_LENGTH        = 12
SEED                  = 7        ← 3rd anchor; 与 §7.9.2 (seed=42) / §7.9.2' (seed=0) 不同
NUM_ENVS              = 6
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM         = False
UPDATES_PER_STEP      = 1
DROPOUT_RATE          = 0.0

benchmark             : single_u15_cross_tgt15
geometry              : cross_stream
total_steps           : 1,000,000
expected obs_dim      : 12 * 12 = 144 (per train_config.txt across all baselines)
run_root              : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_7
§7.9.2  k=12 s42 PASS : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_42
§7.9.2' k=12 s0  RES  : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k12/seed_0
§7.8    k=8  s42 PASS : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42
§7.9.1' k=8  

## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest / baseline 就位


In [4]:
# Flow file
fp = Path(X_FLOW_PATH)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')


[OK] flow file: wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy  (230.4 MB)


In [5]:
!python -u -m scripts.validate_arrival_v2_candidate



[undiscounted]
fast_success       144.473
slow_success       142.023
unsafe_success      91.048
timeout_near      -149.800
timeout_far       -229.800
late_oob          -274.500
fast_oob          -280.325
mid_oob           -319.450

[discounted_gamma_0.995]
fast_success        88.578
slow_success        54.683
unsafe_success      34.857
timeout_near       -39.480
timeout_far        -58.269
late_oob          -103.800
mid_oob           -190.006
fast_oob          -212.555

[discounted_shortcut] safe=69.330 risky=62.361

PASS: arrival_v2 candidate pre-integration gates passed.


In [6]:
!python -u -m pytest tests/test_reward_objective.py -q


...............                                                          [100%]
15 passed in 19.56s


In [7]:
if not X_MANIFEST_PATH.exists():
    !python -u -m scripts.generate_standard_benchmarks --benchmarks {X_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not X_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {X_MANIFEST_PATH}')
print(f'[OK] manifest ready: {X_MANIFEST_PATH}')


[OK] manifest ready: benchmarks/single_u15_cross_tgt15.json


In [8]:
# 检查 8 个对比 baseline 是否就位
for label, root, ref_final, ref_oob in [
    ('§7.9.2  k12 s42 PASS-PLAT   ', X_K12_S42_ROOT, 0.900, 0.100),
    ("§7.9.2' k12 s0  RESCUE      ", X_K12_S0_ROOT,  0.900, 0.100),
    ('§7.8    k8  s42 PASS        ', X_K8_S42_ROOT,  0.900, 0.100),
    ("§7.9.1' k8  s0  PARTIAL     ", X_K8_S0_ROOT,   0.500, 0.133),
    ("§7.9.1'' k8 s7  BORDER ← main", X_K8_S7_ROOT,  0.867, 0.133),
    ('§7.6.4  k4  s42 floor       ', X_K4_S42_ROOT,  0.100, 0.667),
    ('§7.7.1  k4  s0  sister      ', X_K4_S0_ROOT,   0.400, 0.200),
    ('§7.1    s1  k4 upper        ', X_S1_UPPER_ROOT, 0.900, 0.100),
]:
    fp = root / 'results' / 'final_eval.json'
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        oob = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        match = '✓' if abs(f - ref_final) < 0.05 and abs(oob - ref_oob) < 0.05 else '✗ mismatch'
        print(f'[OK] {label}: final={f:.4f}  oob={oob:.4f}  counts={c}  {match}')
    else:
        print(f'[WARN] {label}: {fp} 不存在 (后续 §5 diff 会回退到 report 转载值)')


[OK] §7.9.2  k12 s42 PASS-PLAT   : final=0.9000  oob=0.1000  counts={'goal': 27, 'out_of_bounds': 3}  ✓
[OK] §7.9.2' k12 s0  RESCUE      : final=0.9000  oob=0.1000  counts={'goal': 27, 'out_of_bounds': 3}  ✓
[OK] §7.8    k8  s42 PASS        : final=0.9000  oob=0.1000  counts={'goal': 27, 'out_of_bounds': 3}  ✓
[OK] §7.9.1' k8  s0  PARTIAL     : final=0.5000  oob=0.1333  counts={'timeout': 11, 'goal': 15, 'out_of_bounds': 4}  ✓
[OK] §7.9.1'' k8 s7  BORDER ← main: final=0.8667  oob=0.1333  counts={'goal': 26, 'out_of_bounds': 4}  ✓
[OK] §7.6.4  k4  s42 floor       : final=0.1000  oob=0.6667  counts={'out_of_bounds': 20, 'timeout': 7, 'goal': 3}  ✓
[OK] §7.7.1  k4  s0  sister      : final=0.4000  oob=0.2000  counts={'timeout': 12, 'out_of_bounds': 6, 'goal': 12}  ✓
[OK] §7.1    s1  k4 upper        : final=0.9000  oob=0.1000  counts={'goal': 27, 'out_of_bounds': 3}  ✓


## 4. Train — single_cross_s0 + history k=12 + seed=7 (1.0M, skip/resume)


In [9]:
x_state_path = X_RUN_ROOT / 'trainer_state.json'
if x_state_path.exists():
    x_state = json.loads(x_state_path.read_text(encoding='utf-8'))
    x_current_step = int(x_state.get('env_step', 0))
else:
    x_current_step = 0
print(f'[state] X env_step = {x_current_step:,} / target {X_TOTAL_STEPS:,}')

if x_current_step >= X_TOTAL_STEPS:
    print(f'[skip] X already trained to {x_current_step:,} >= {X_TOTAL_STEPS:,}')
elif x_current_step > 0:
    print(f'[resume] X continuing from {x_current_step:,} -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(X_RUN_ROOT)} \
        --total-steps {X_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] X fresh start -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {X_FLOW_PATH} \
        --task-geometry {X_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {X_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(X_RUN_ROOT)} \
        --checkpoint-dir {str(X_CKPT_ROOT)}


[state] X env_step = 0 / target 1,000,000
[train] X fresh start -> 1,000,000
[train] episode=5 step=1134 return=-392.07 success=False time=94.2s geometry=cross_stream history=12
[train] episode=10 step=1854 return=-386.00 success=False time=80.0s geometry=cross_stream history=12
[train] episode=15 step=2736 return=-408.78 success=False time=83.1s geometry=cross_stream history=12
[train] episode=20 step=3060 return=-416.75 success=False time=99.8s geometry=cross_stream history=12
[train] episode=25 step=3936 return=-429.27 success=False time=72.1s geometry=cross_stream history=12
[train] episode=30 step=4590 return=-453.34 success=False time=109.3s geometry=cross_stream history=12
[train] episode=35 step=5130 return=-265.15 success=False time=26.0s geometry=cross_stream history=12 | q1=0.363 actor=0.897 alpha=0.199
[train] episode=40 step=5478 return=-314.28 success=False time=22.5s geometry=cross_stream history=12 | q1=1.988 actor=2.086 alpha=0.195
[train] episode=45 step=6444 return=-

## 5. Summary + gate


In [10]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'
    train_config_path = run_root / 'results' / 'train_config.txt'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    history_from_config = 'NA'
    obs_dim_from_config = 'NA'
    if train_config_path.exists():
        for ln in train_config_path.read_text(encoding='utf-8').splitlines():
            s = ln.strip()
            if s.startswith('history_length='):
                history_from_config = s.split('=', 1)[1]
            elif s.startswith('obs_dim='):
                obs_dim_from_config = s.split('=', 1)[1]

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    mean_all = float(df['eval_success_rate'].mean()) if len(df) else 0.0
    n_evals_with_success = int((df['eval_success_rate'] > 0).sum()) if len(df) else 0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s0 / k=12 / seed={SEED} / vanilla / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  full-traj mean        : {mean_all:.4f}   (39 evals)")
    print(f"  n_evals_with_success  : {n_evals_with_success} / {len(df)}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {obs_dim_from_config}   (expect 12*12=144)")
    print(f"  history_length        : {history_from_config}   (from train_config.txt)")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': int(history_from_config) if history_from_config != 'NA' else HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'algorithm': 'sac_vanilla',
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'mean_success_full_trajectory': mean_all,
        'n_evals_with_success': n_evals_with_success,
        'n_evals_total': len(df),
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

x_summary = summarize_phase(
    X_RUN_ROOT,
    X_TOTAL_STEPS,
    'SINGLE_CROSS_S0_K12_SEED7',
    'single_cross_s0_k12_seed7_gate_summary.json',
)
X_PASS = x_summary['all_pass']
print()
print(f'X_PASS = {X_PASS}')


SINGLE_CROSS_S0_K12_SEED7  (arrival_v2 / s0 / k=12 / seed=7 / vanilla / 1,000,000 steps)
----------------------------------------------------------------------------------------------------
  final_success_rate    : 0.8333   gate >= 0.85
  peak_success_rate     : 0.9000   @ 275,004
  last100k_mean_success : 0.8583   gate >= 0.8100
  full-traj mean        : 0.7504   (39 evals)
  n_evals_with_success  : 36 / 39
  final_oob_rate        : 0.1667   gate <= 0.10
  obs_dim               : 144   (expect 12*12=144)
  history_length        : 12   (from train_config.txt)
  context_obs           : True
  timeout_bootstrap     : terminal
  termination           : {'goal': 25, 'out_of_bounds': 5}

[last 16 eval rows]
 env_step  eval_success_rate  eval_return  eval_safety_cost  eval_time_s  eval_progress_ratio
   600000           0.900000    85.351659         10.815727    78.910000             0.835362
   625002           0.866667    81.008597         10.113876    88.063333             0.831426
   65

## 6. Cross-seed closure verdict — k=12 seed=7 (3rd anchor) vs §7.9.1'' k=8 s7 + §7.9.2/2' k=12 {s42, s0} + 全 8-run 对照 + 3-seed σ_final closure


In [11]:
import math


def read_baseline(root: Path, ref_final: float, ref_oob: float, ref_mean: float = None):
    fp = root / 'results' / 'final_eval.json'
    log = root / 'results' / 'eval_log.csv'
    out = {'final': ref_final, 'oob': ref_oob, 'mean': ref_mean, 'peak': None, 'peak_step': None, 'source': 'report'}
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        out['final'] = f
        out['oob'] = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        out['source'] = 'on-disk'
    if log.exists():
        dlog = pd.read_csv(log)
        out['mean'] = float(dlog['eval_success_rate'].mean())
        out['peak'] = float(dlog['eval_success_rate'].max())
        out['peak_step'] = int(dlog.loc[dlog['eval_success_rate'].idxmax(), 'env_step'])
    return out


def sample_stats(xs):
    n = len(xs)
    mu = sum(xs) / n
    var = sum((x - mu) ** 2 for x in xs) / (n - 1) if n > 1 else 0.0
    return mu, math.sqrt(var)


print('=' * 110)
print("SINGLE_CROSS — k=12 cross-seed closure verdict (seed=7 = 3rd anchor; main contrast = §7.9.1'' k=8 s7)")
print('-' * 110)

# 本 run (k=12 seed=7)
k12s7_final = float(x_summary['final_success_rate'])
k12s7_oob   = float(x_summary['final_oob_rate'])
k12s7_peak  = float(x_summary['peak_success_rate'])
k12s7_peak_step = int(x_summary['peak_step'])
k12s7_mean  = float(x_summary.get('mean_success_full_trajectory', 0.0))
k12s7_nsucc = int(x_summary.get('n_evals_with_success', 0))
k12s7_ntot  = int(x_summary.get('n_evals_total', 0))

# 8 个 baseline
b_k12_s42 = read_baseline(X_K12_S42_ROOT, 0.900, 0.100, 0.652)  # §7.9.2 anchor PASS-PLATEAU
b_k12_s0  = read_baseline(X_K12_S0_ROOT,  0.900, 0.100, 0.525)  # §7.9.2' CROSS-SEED-RESCUE
b_k8_s42  = read_baseline(X_K8_S42_ROOT,  0.900, 0.100, 0.636)  # §7.8 anchor PASS
b_k8_s0   = read_baseline(X_K8_S0_ROOT,   0.500, 0.133, 0.260)  # §7.9.1' PARTIAL
b_k8_s7   = read_baseline(X_K8_S7_ROOT,   0.867, 0.133, 0.518)  # §7.9.1'' BORDER — same-seed 主对照
b_k4_s42  = read_baseline(X_K4_S42_ROOT,  0.100, 0.667, 0.221)  # §7.6.4 floor
b_k4_s0   = read_baseline(X_K4_S0_ROOT,   0.400, 0.200, 0.218)  # §7.7.1 sister
b_s1      = read_baseline(X_S1_UPPER_ROOT, 0.900, 0.100, 0.497) # §7.1 upper

# Labels with single-quotes (§7.9.X') need to be pre-defined to avoid f-string quote conflict.
label_k8_s0   = "vanilla s0_k8 s0  (§7.9.1' PARTIAL)"
label_k8_s7   = "vanilla s0_k8 s7  (§7.9.1'' BORDER ← main)"
label_k12_s0  = "vanilla s0_k12 s0  (§7.9.2' RESCUE)"

print()
print(f'{"config":<48}{"final":>10}{"mean":>10}{"oob":>10}{"peak":>10}{"peak@":>14}')
print('-' * 110)
print(f'{"vanilla s1_k4 s42 (§7.1 upper ref)":<48}{b_s1["final"]:>10.4f}{(b_s1["mean"] or float("nan")):>10.4f}{b_s1["oob"]:>10.4f}{(b_s1["peak"] or float("nan")):>10.4f}{(b_s1["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k4 s42 (§7.6.4 FAIL floor)":<48}{b_k4_s42["final"]:>10.4f}{(b_k4_s42["mean"] or float("nan")):>10.4f}{b_k4_s42["oob"]:>10.4f}{(b_k4_s42["peak"] or float("nan")):>10.4f}{(b_k4_s42["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k4 s0  (§7.7.1 sister)":<48}{b_k4_s0["final"]:>10.4f}{(b_k4_s0["mean"] or float("nan")):>10.4f}{b_k4_s0["oob"]:>10.4f}{(b_k4_s0["peak"] or float("nan")):>10.4f}{(b_k4_s0["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k8 s42 (§7.8 anchor PASS)":<48}{b_k8_s42["final"]:>10.4f}{(b_k8_s42["mean"] or float("nan")):>10.4f}{b_k8_s42["oob"]:>10.4f}{(b_k8_s42["peak"] or float("nan")):>10.4f}{(b_k8_s42["peak_step"] or 0):>14,}')
print(f'{label_k8_s0:<48}{b_k8_s0["final"]:>10.4f}{(b_k8_s0["mean"] or float("nan")):>10.4f}{b_k8_s0["oob"]:>10.4f}{(b_k8_s0["peak"] or float("nan")):>10.4f}{(b_k8_s0["peak_step"] or 0):>14,}')
print(f'{label_k8_s7:<48}{b_k8_s7["final"]:>10.4f}{(b_k8_s7["mean"] or float("nan")):>10.4f}{b_k8_s7["oob"]:>10.4f}{(b_k8_s7["peak"] or float("nan")):>10.4f}{(b_k8_s7["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k12 s42 (§7.9.2 PASS-PLAT)":<48}{b_k12_s42["final"]:>10.4f}{(b_k12_s42["mean"] or float("nan")):>10.4f}{b_k12_s42["oob"]:>10.4f}{(b_k12_s42["peak"] or float("nan")):>10.4f}{(b_k12_s42["peak_step"] or 0):>14,}')
print(f'{label_k12_s0:<48}{b_k12_s0["final"]:>10.4f}{(b_k12_s0["mean"] or float("nan")):>10.4f}{b_k12_s0["oob"]:>10.4f}{(b_k12_s0["peak"] or float("nan")):>10.4f}{(b_k12_s0["peak_step"] or 0):>14,}')
print(f'{"vanilla s0_k12 s7  (THIS RUN, 3rd anchor)":<48}{k12s7_final:>10.4f}{k12s7_mean:>10.4f}{k12s7_oob:>10.4f}{k12s7_peak:>10.4f}{k12s7_peak_step:>14,}')
print('=' * 110)

# Δ 行 — 主对照 vs k=8 s7 (same seed) + vs k=12 s42 / s0 (cross seed within k=12)
delta_final_vs_k8_s7   = k12s7_final - b_k8_s7['final']
delta_mean_vs_k8_s7    = k12s7_mean  - (b_k8_s7['mean'] or 0.0)
delta_oob_vs_k8_s7     = k12s7_oob   - b_k8_s7['oob']
delta_final_vs_k12_s42 = k12s7_final - b_k12_s42['final']
delta_final_vs_k12_s0  = k12s7_final - b_k12_s0['final']
delta_final_vs_s1      = k12s7_final - b_s1['final']

print()
print(f'{"contrast":<60}{"Δ final":>12}{"Δ mean":>12}{"Δ oob":>12}')
print('-' * 110)
print(f'{"k=12 s7 vs k=8 s7  (cross-history, same seed=7) ← main":<60}'
      f'{delta_final_vs_k8_s7:>+12.4f}{delta_mean_vs_k8_s7:>+12.4f}{delta_oob_vs_k8_s7:>+12.4f}')
print(f'{"k=12 s7 vs k=12 s42 (cross-seed within k=12 anchor)":<60}'
      f'{delta_final_vs_k12_s42:>+12.4f}{(k12s7_mean - (b_k12_s42["mean"] or 0)):>+12.4f}{k12s7_oob - b_k12_s42["oob"]:>+12.4f}')
print(f'{"k=12 s7 vs k=12 s0  (cross-seed within k=12 RESCUE)":<60}'
      f'{delta_final_vs_k12_s0:>+12.4f}{(k12s7_mean - (b_k12_s0["mean"] or 0)):>+12.4f}{k12s7_oob - b_k12_s0["oob"]:>+12.4f}')
print(f'{"k=12 s7 vs s1_k4 s42 (gap-to-upper)":<60}'
      f'{delta_final_vs_s1:>+12.4f}'
      f'{(k12s7_mean - (b_s1["mean"] or 0)):>+12.4f}'
      f'{k12s7_oob - b_s1["oob"]:>+12.4f}')
print('=' * 110)
print()
print(f'k=12 seed=7 evals_with_success: {k12s7_nsucc} / {k12s7_ntot}  '
      f'(k=12 s42 was 35/39 PASS-PLATEAU, k=12 s0 was 32/39 RESCUE, k=8 s7 was 31/39 BORDER)')

# ====== 3-seed closure: thesis-grade σ_final 计算 ======
print()
print('=' * 110)
print('3-seed σ_final closure (k=12 × {42, 0, 7}) — thesis-grade target σ_final ≤ 0.10')
print('-' * 110)
k12_3seed_finals = [b_k12_s42['final'], b_k12_s0['final'], k12s7_final]
k12_3seed_oobs   = [b_k12_s42['oob'],   b_k12_s0['oob'],   k12s7_oob]
k12_3seed_means  = [b_k12_s42['mean'] or 0.0, b_k12_s0['mean'] or 0.0, k12s7_mean]

mu_final, sigma_final = sample_stats(k12_3seed_finals)
mu_oob,   sigma_oob   = sample_stats(k12_3seed_oobs)
mu_mean,  sigma_mean  = sample_stats(k12_3seed_means)

# 与 k=8 三 seed 对比 (42 / 0 / 7)
k8_3seed_finals = [b_k8_s42['final'], b_k8_s0['final'], b_k8_s7['final']]
k8_3seed_oobs   = [b_k8_s42['oob'],   b_k8_s0['oob'],   b_k8_s7['oob']]
k8_3seed_means  = [b_k8_s42['mean'] or 0.0, b_k8_s0['mean'] or 0.0, b_k8_s7['mean'] or 0.0]
mu_k8_final, sigma_k8_final = sample_stats(k8_3seed_finals)
mu_k8_oob,   sigma_k8_oob   = sample_stats(k8_3seed_oobs)
mu_k8_mean,  sigma_k8_mean  = sample_stats(k8_3seed_means)

print(f'{"metric":<20}{"k=8 (3 seed)":>24}{"k=12 (3 seed)":>24}{"Δ (k=12 − k=8)":>26}')
print('-' * 110)
print(f'{"final mean ± σ":<20}{f"{mu_k8_final:.4f} ± {sigma_k8_final:.4f}":>24}{f"{mu_final:.4f} ± {sigma_final:.4f}":>24}{f"{mu_final - mu_k8_final:+.4f} / σΔ {sigma_final - sigma_k8_final:+.4f}":>26}')
print(f'{"oob   mean ± σ":<20}{f"{mu_k8_oob:.4f} ± {sigma_k8_oob:.4f}":>24}{f"{mu_oob:.4f} ± {sigma_oob:.4f}":>24}{f"{mu_oob - mu_k8_oob:+.4f} / σΔ {sigma_oob - sigma_k8_oob:+.4f}":>26}')
print(f'{"mean39 mean ± σ":<20}{f"{mu_k8_mean:.4f} ± {sigma_k8_mean:.4f}":>24}{f"{mu_mean:.4f} ± {sigma_mean:.4f}":>24}{f"{mu_mean - mu_k8_mean:+.4f} / σΔ {sigma_mean - sigma_k8_mean:+.4f}":>26}')
print('-' * 110)
THESIS_SIGMA_TARGET = 0.10
thesis_grade = (sigma_final <= THESIS_SIGMA_TARGET) and all(f >= PASS_FINAL_SUCCESS for f in k12_3seed_finals)
print(f'thesis-grade closure (k=12 三 seed 全 strict-PASS AND σ_final ≤ {THESIS_SIGMA_TARGET}): {thesis_grade}')
print('=' * 110)

# Seed=7 跨 history 轨迹（k=4 未测；只有 k=8 / k=12 两点）
print()
print(f'seed=7 cross-history trajectory: k=4: n/a (untested per §7.9.6), k=8: {b_k8_s7["final"]:.4f} (BORDER), k=12: {k12s7_final:.4f} (this run)')
print(f'seed=0 cross-history trajectory (reference): k=4: {b_k4_s0["final"]:.4f}, k=8: {b_k8_s0["final"]:.4f}, k=12: {b_k12_s0["final"]:.4f} (§7.9.3 monotonic phase transition)')

# ====== Cross-seed closure verdict — 5-tier ======
strict_pass     = (k12s7_final >= PASS_FINAL_SUCCESS) and (k12s7_oob <= PASS_OOB_RATE)
borderline_pass = (k12s7_final >= PASS_FINAL_SUCCESS) and (PASS_OOB_RATE < k12s7_oob <= BORDERLINE_OOB_RATE)
# UNEXPECTED-DECAY 优先于 PERSISTENT-STALL（cross-history regression 信号比 stall 强）
unexpected_decay = (not strict_pass and not borderline_pass) and (
    (k12s7_final < b_k8_s7['final'] - 0.05) or (0.40 <= k12s7_final < 0.50)
)
persistent_stall = (not strict_pass and not borderline_pass and not unexpected_decay) and (
    (0.50 <= k12s7_final < PASS_FINAL_SUCCESS) and (abs(k12s7_final - b_k8_s7['final']) <= 0.20)
)
collapse = k12s7_final < 0.40

if strict_pass:
    verdict_tier = 'CROSS-SEED-CLOSURE-3ANCHOR'
    verdict = (
        f'CROSS-SEED-CLOSURE-3ANCHOR — k=12 s7 strict 5/5 PASS (final={k12s7_final:.3f}≥0.85, OOB={k12s7_oob:.3f}≤0.10); '
        f'k=12 三 seed {{42, 0, 7}} = {{{b_k12_s42["final"]:.3f}, {b_k12_s0["final"]:.3f}, {k12s7_final:.3f}}}, 3-seed σ_final={sigma_final:.4f} '
        f'({"≤" if sigma_final <= THESIS_SIGMA_TARGET else ">"} thesis target {THESIS_SIGMA_TARGET}); '
        f'vs k=8 s7 same-seed Δfinal=+{delta_final_vs_k8_s7:.3f} (BORDER → strict PASS, OOB clear). '
        f"§7.9.6 唯一 open disclaimer (k=12 third anchor) CLOSED; arrival_v2 prototype §7.9 thesis-grade closure 达成; "
        f'paper rebuttal 可直接 cite "k=12 cross-seed (3 seed) σ_final={sigma_final:.3f}". '
        f'§7.9.4 主张升格「k=12 是 cross-seed sweet spot, 3-seed thesis-grade」; '
        f'§8 P0 SAC variance reduction motivation 维持 polish-only 定位 (3-seed σ_final 已自然达标)'
    )
elif borderline_pass:
    verdict_tier = 'CROSS-SEED-BORDERLINE-CLOSURE'
    verdict = (
        f'CROSS-SEED-BORDERLINE-CLOSURE — k=12 s7 final={k12s7_final:.3f}≥0.85 ✓ 但 OOB={k12s7_oob:.3f}>0.10 ✗ (≤0.135 BORDER); '
        f'final 维度 closure 成立 (vs k=8 s7 Δfinal=+{delta_final_vs_k8_s7:.3f}); '
        f'但 OOB ≈ k=8 s7 (=0.133) 跨 history 不变 → OOB residual variance 与 actor information capacity 解耦, 是 SAC noise 不是 history 问题. '
        f'3-seed σ_final={sigma_final:.4f}; OOB σ={sigma_oob:.4f}. '
        f'§7.9.4 主张可写「final cross-seed 闭合, OOB BORDER tail 与 history 解耦」; '
        f'§8 P0 variance reduction priority elevated — 用于压低 OOB tail；history 路径 thesis-defensible'
    )
elif unexpected_decay:
    verdict_tier = 'UNEXPECTED-DECAY'
    verdict = (
        f'UNEXPECTED-DECAY — k=12 s7 final={k12s7_final:.3f} ≤ k=8 s7 final={b_k8_s7["final"]:.3f} − 0.05 = {b_k8_s7["final"]-0.05:.3f}; '
        f'over-stale history 在 seed=7 上反害, history extension 不再 monotonic across seeds. '
        f"与 §7.9.2 (s42) / §7.9.2' (s0) k=12 PASS 形成 cross-seed conflict; §7.9.4 主张需 reframe; "
        f'audit recommended (eval_log.csv 中段 trajectory + termination_counts 看 OOB 模式); '
        f'强烈推荐 pivot §8 P0 variance reduction 代替进一步 history extension'
    )
elif persistent_stall:
    verdict_tier = 'PERSISTENT-STALL'
    verdict = (
        f'PERSISTENT-STALL — k=12 s7 final={k12s7_final:.3f} ∈ [0.5, 0.85), 与 k=8 s7 final={b_k8_s7["final"]:.3f} 在 0.20 范围内; '
        f'seed=7 stall 跨 history 持续 (k=8: {b_k8_s7["final"]:.3f}, k=12: {k12s7_final:.3f}). '
        f"与 §7.9.2 (s42) / §7.9.2' (s0) k=12 strict PASS 形成 cross-seed conflict; §7.9.4 主张需 reframe; "
        f'§8 P0 variance reduction 必要'
    )
elif collapse:
    verdict_tier = 'COLLAPSE'
    verdict = (
        f'COLLAPSE — k=12 s7 final={k12s7_final:.3f}<0.40, 严重下降. '
        f'必须 audit: trainer_state grad-norm / actor-critic loss curve / obs_dim 排除 training bug; '
        f"若非 bug, thesis claim 严重 reframe; 与 §7.9.2 / §7.9.2' cross-seed PASS 直接冲突"
    )
else:
    verdict_tier = 'UNCLASSIFIED'
    verdict = (
        f'UNCLASSIFIED — final={k12s7_final:.3f} OOB={k12s7_oob:.3f}; '
        f"5-tier schema 未触发任何明确条件; 需 manual 审视 (本 verdict 应被视为 bug 报告 — 类似 §7.9.1'' k=8 s7 4-tier 漏 BORDER 教训)"
    )

print()
print(f'>>> verdict_tier: {verdict_tier}')
print(f'>>> verdict     : {verdict}')

# 落盘 cross-seed closure summary
mono_out = {
    'experiment': 'arrival_v2_s0_cross_k12_seed7_crossseed_closure_3anchor',
    'seed': SEED,
    'benchmark': X_BENCHMARK_KEY,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'total_steps': X_TOTAL_STEPS,
    'algorithm': 'sac_vanilla',
    'cli_diff_vs_k12_anchor_seed42': '--seed 42 → 7',
    'cli_diff_vs_k8_seed7_main_contrast': '--history-length 8 → 12',
    'results': {
        'k12_s7_thisrun': {
            'final': k12s7_final, 'mean': k12s7_mean, 'oob': k12s7_oob,
            'peak': k12s7_peak, 'peak_step': k12s7_peak_step,
            'n_evals_with_success': k12s7_nsucc, 'n_evals_total': k12s7_ntot,
        },
        'k12_s42_anchor_pass_plateau': b_k12_s42,   # §7.9.2
        'k12_s0_cross_seed_rescue': b_k12_s0,       # §7.9.2'
        'k8_s42_anchor_pass': b_k8_s42,             # §7.8
        'k8_s0_partial': b_k8_s0,                   # §7.9.1'
        'k8_s7_borderline_pass_main_contrast': b_k8_s7,  # §7.9.1''
        'k4_s42_floor': b_k4_s42,                   # §7.6.4
        'k4_s0_sister': b_k4_s0,                    # §7.7.1
        's1_k4_s42_upper': b_s1,                    # §7.1
    },
    'delta_vs_main_contrast_k8_s7_same_seed': {
        'final_pp': round(delta_final_vs_k8_s7 * 100, 2),
        'mean_pp':  round(delta_mean_vs_k8_s7  * 100, 2),
        'oob_pp':   round(delta_oob_vs_k8_s7   * 100, 2),
    },
    'delta_vs_k12_anchor_seed42': {'final_pp': round(delta_final_vs_k12_s42 * 100, 2)},
    'delta_vs_k12_sister_seed0':  {'final_pp': round(delta_final_vs_k12_s0  * 100, 2)},
    'seed7_cross_history_trajectory': {
        'k4':  None,  # untested per §7.9.6
        'k8':  b_k8_s7['final'],
        'k12': k12s7_final,
    },
    'seed0_cross_history_trajectory_reference': {
        'k4':  b_k4_s0['final'],
        'k8':  b_k8_s0['final'],
        'k12': b_k12_s0['final'],
    },
    'k12_3seed_closure': {
        'seeds': [42, 0, 7],
        'finals': k12_3seed_finals,
        'oobs':   k12_3seed_oobs,
        'means':  k12_3seed_means,
        'final_mean': mu_final,
        'final_sigma': sigma_final,
        'oob_mean': mu_oob,
        'oob_sigma': sigma_oob,
        'mean_mean': mu_mean,
        'mean_sigma': sigma_mean,
        'thesis_sigma_target': THESIS_SIGMA_TARGET,
        'thesis_grade_closure': bool(thesis_grade),
    },
    'k8_3seed_reference': {
        'seeds': [42, 0, 7],
        'finals': k8_3seed_finals,
        'final_mean': mu_k8_final,
        'final_sigma': sigma_k8_final,
        'oob_sigma': sigma_k8_oob,
        'mean_sigma': sigma_k8_mean,
    },
    'verdict_tier': verdict_tier,
    'all_pass': bool(x_summary['all_pass']),
    'verdict': verdict,
}
out_dir = Path('experiments/arrival_v2_prototype/s0_cross_k12_seed7_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(mono_out, indent=2), encoding='utf-8')
print(f'\n[saved] {out_path}')


SINGLE_CROSS — k=12 cross-seed closure verdict (seed=7 = 3rd anchor; main contrast = §7.9.1'' k=8 s7)
--------------------------------------------------------------------------------------------------------------

config                                               final      mean       oob      peak         peak@
--------------------------------------------------------------------------------------------------------------
vanilla s1_k4 s42 (§7.1 upper ref)                  0.9000    0.4966    0.1000    0.9000       725,004
vanilla s0_k4 s42 (§7.6.4 FAIL floor)               0.1000    0.2205    0.6667    0.3667       975,000
vanilla s0_k4 s0  (§7.7.1 sister)                   0.4000    0.2179    0.2000    0.5333       625,002
vanilla s0_k8 s42 (§7.8 anchor PASS)                0.9000    0.6359    0.1000    0.9000       475,002
vanilla s0_k8 s0  (§7.9.1' PARTIAL)                 0.5000    0.2598    0.1333    0.5000       925,002
vanilla s0_k8 s7  (§7.9.1'' BORDER ← main)          0.866